# Household and housing questionnaire summaries

Rebuild Table 1 and Table S4 through auditable address linkage. Questionnaire variables are descriptive and are not covariates in the primary retrofit models.

**Step:** 05. Household questionnaire tables (Tables 1 and S4).

**Article scope:** the main article and Supplementary Material.

**Inputs:** The retained-sensor register, dwelling assessment workbook, and project survey workbook.

**Outputs:** Table 1 categorical and numeric summaries, Table S4, and a home-level linkage audit.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure this reproduction run

Choose the separate output directory. The default is `results/` in the repository folder; rebuilt files in that directory take precedence over supplied inputs. Use a new directory when testing a changed input.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

## Import the required libraries

These imports belong to the active notebook kernel. Install `requirements.txt` in that same environment if an import fails.

In [ ]:
import re
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from shared import DATA, crosswalk, output_root, dump_json

# Short name -> exact column header in the assessment survey (original spelling kept).
FIELDS = {
    'occupants':'Number of occupants in the dwelling',
    'thermostat':'Room stat temperature (C or unknown)',
    'showers':'On average how many times do you shower per week (if known)',
    'baths':'On average, how many times do you bathe per week (if known)',
    'gas':'Kitchen appliances.Gas cooker', 'electric':'Kitchen appliances.Electric cooker',
    'tenure':'Property tenure', 'pets':'Does the household have a pet?',
    'smoking':'Does anyone in the household smoke or vape?',
    'smoking_location':'If yes, inside or outside the house?', 'shower_type':'Shower type',
    'thermostat_location':'Room stat location', 'overheating':'Do you experience overheating during the summer months?',
    'air_conditioning':'Does the resident have an air conditioning unit?', 'respondent_status':'Status of respondent'
}

## Normalize

Normalize address IDs only: trim whitespace, standardize case, and avoid adding a decimal suffix to integer IDs. An empty or unknown key remains missing.

In [ ]:
def normalize(value):
    """Normalize an address ID: trim, lower-case, and write integers without '.0'.

    Empty or unknown ('???') IDs become None.
    """
    if value is None or str(value).strip() in ['', '???']:
        return None
    if isinstance(value, (float, int)):
        return str(int(value)) if value == int(value) else str(value)
    return str(value).strip().lower()

## Records

Index worksheet rows by their address ID while retaining row numbers. Keeping multiple matches visible lets the later linkage assertions detect ambiguous records.

In [ ]:
def records(sheet, header):
    """Index worksheet rows by normalized address ID.

    Returns {address_id: [(row_number, {column: value}), ...]}. More than one
    entry for an ID means an ambiguous match, which is rejected later.
    """
    headers = [str(c.value).strip() if c.value is not None else None for c in sheet[header]]
    rows = {}
    for r in range(header + 1, sheet.max_row + 1):
        if sheet.cell(r, 1).value is None:
            continue
        row = {h: sheet.cell(r, i + 1).value for i, h in enumerate(headers) if h}
        key = normalize(row['Address ID'])
        if key is not None:
            rows.setdefault(key, []).append((r, row))
    return rows

## Merged value

Recover the top-left value of a merged register cell when a retained sensor occupies a continuation row. This preserves the registered residence relationship.

In [ ]:
def merged_value(sheet, row, col):
    """Cell value; for a cell inside a merged range, the range's top-left value."""
    value = sheet.cell(row, col).value
    if value is None:
        for span in sheet.merged_cells.ranges:
            if span.min_row <= row <= span.max_row and span.min_col <= col <= span.max_col:
                return sheet.cell(span.min_row, span.min_col).value
    return value

## Number

Accept a plain numeric response, optionally with a temperature unit. Ambiguous text becomes missing instead of being guessed.

In [ ]:
def number(value):
    """Parse a plain number, optionally followed by a degree-C unit; otherwise NaN."""
    m = re.fullmatch(r'\s*([+-]?\d+(?:\.\d+)?)\s*(?:°?\s*[cC])?\s*', str(value))
    return float(m.group(1)) if m else np.nan

## Read the register and the two questionnaire sources

Load cached Excel values and index responses by normalized address ID. Link each retained sensor through the register; do not use a legacy home number as a join key.

In [ ]:
out = output_root(OUTPUT)
# data_only=True reads the cached cell values instead of formulas.
register = load_workbook(DATA / 'Brockmoor_home_info.xlsx', data_only=True).active
assessment = load_workbook(DATA / 'Survey.xlsx', data_only=True).active
project = load_workbook(DATA / 'Dudley-NZN-Sensor-Retrofit-Data - UoB.xlsx', data_only=True)['Survey Responses']
assessment_records = records(assessment, 1)   # header in row 1
project_records = records(project, 2)         # header in row 2

## Resolve household records and housing descriptions

Require at most one matching record in each survey. Preserve missing responses. Respondent tenure status and legal tenure are different fields: conflicting tenure evidence is marked NR. Housing descriptions follow explicit source answers, including the recorded ground-floor-flat description. Literal Excel column names retain their original spelling so that lookup remains exact.

In [ ]:
# Source answer -> category shown in Table S4 ('NR' = not recorded).
DWELLING_TYPES = {'Semi-Detached': 'Semi-detached', 'End terrace': 'End-terrace', 'Mid Terraced': 'Mid-terrace',
                  'First Floor Flat': 'First-floor flat', 'Ground Floor Flat': 'Ground-floor flat'}
RESPONDENT_STATUS = {'Tenant': 'Rented (sector NR)', 'Homeowner': 'Owner-occupied'}
TENURE_CODES = {'OO': 'Owner-occupied', 'SH': 'Social rented', 'PR': 'Private rented'}
RESIDENCE_YEARS = {'Less than 1 year': '<1', '1 - 3 years': '1–3', '3 - 5 years': '3–5',
                   '5 - 10 years': '5–10', 'More than 10 years': '>10'}


def integer_answer(answers, field):
    """A whole-number answer as text, otherwise 'NR'."""
    v = number(answers.get(field))
    return str(int(v)) if np.isfinite(v) and v == int(v) else 'NR'


qs = []        # assessment answers per home (Table 1)
housing = []   # housing description per home (Table S4)
linkage = []   # source rows used for each home
for h in crosswalk().itertuples():
    # Sensor -> register row -> address ID (merged register cells are resolved).
    register_rows = [r for r in range(2, register.max_row + 1) if register.cell(r, 2).value == h.sensor_id]
    assert len(register_rows) == 1
    key = normalize(merged_value(register, register_rows[0], 1))

    # At most one record per survey; a missing record stays missing.
    aa = assessment_records.get(key, []) if key else []
    pp = project_records.get(key, []) if key else []
    assert len(aa) <= 1 and len(pp) <= 1
    arow, a = aa[0] if aa else (None, {})
    prow, p = pp[0] if pp else (None, {})

    q = dict(home=h.home, home_id=h.home_id, sensor_id=h.sensor_id, assessment_row=arow)
    q.update({name: a.get(field) for name, field in FIELDS.items()})
    qs.append(q)

    # Dwelling type, including the explicitly recorded ground-floor flat.
    dtype = DWELLING_TYPES.get(p.get('CLEANED Archetype'), 'NR')
    if dtype == 'NR' and a.get('Estimated roof area (in meters squared)') == 'Ground floor flat':
        dtype = 'Ground-floor flat'

    # Tenure: the project-survey code if recorded, otherwise the respondent
    # status; contradictory evidence is reported as NR.
    status = RESPONDENT_STATUS.get(a.get('Status of respondent'), 'NR')
    tenure = TENURE_CODES.get(p.get('Tenure'), 'NR')
    conflict = ((status == 'Owner-occupied' and tenure in ['Social rented', 'Private rented'])
                or (status.startswith('Rented') and tenure == 'Owner-occupied'))
    tenure = 'NR' if conflict else tenure if tenure != 'NR' else status

    duration = RESIDENCE_YEARS.get(p.get('How long have you lived at this property?'), 'NR')
    damp = p.get('(COLLATED) Do you currently have any damp and/ or mould in your home?')
    housing.append({
        'Home': h.home_id,
        'Dwelling type': dtype,
        'Housing tenure': tenure,
        'Bedrooms (n)': integer_answer(p, 'How many bedrooms does this property have?'),
        'Habitable rooms (n)': integer_answer(p, '2.0 Number of Habitable Rooms:'),
        'Length of residence (years)': duration,
        'Damp or mold (self-reported)': damp if damp in ['Yes', 'No'] else 'NR',
    })
    linkage.append(dict(home=h.home, retained_sensor=h.sensor_id, register_row=register_rows[0],
                        assessment_row=arow, project_survey_row=prow, tenure_conflict=conflict))

## Clean missing tokens and form descriptive categories

Treat the source's explicit missing tokens as missing. Parse only unambiguous numeric answers. Gas takes precedence when identifying recorded cooking fuel, and the sole positive smoking answer is checked against the recorded indoor location.

In [ ]:
q = pd.DataFrame(qs)

# Treat the source's explicit missing tokens as missing (this matches pandas'
# Excel missing-value handling, including a literal "n/a").
missing_tokens = {'', 'n/a', 'N/A', 'NA', '#N/A', 'NaN', 'nan', 'NULL', 'null', 'None'}
q = q.apply(lambda col: col.map(lambda v: np.nan if isinstance(v, str) and v in missing_tokens else v))

# Numeric answers; ambiguous text becomes NaN.
for c in ['occupants', 'thermostat', 'showers', 'baths', 'gas', 'electric']:
    q[c] = q[c].map(number)

# Cooking fuel: gas takes precedence when both cookers are recorded.
q['cooking'] = np.where(q.gas.eq(1), 'Gas', np.where(q.electric.eq(1), 'Electric', 'Not recorded'))
loc = (q.thermostat_location.str.strip().str.lower()
       .replace({'hall': 'Hallway or hall', 'hallway': 'Hallway or hall',
                 'living room': 'Living or dining room', 'dining room': 'Living or dining room'}))
smoke = q.smoking.where(~q.smoking.eq('Yes'), 'Yes, indoors')
# The single recorded positive answer was explicitly "Inside".
assert q.loc[q.smoking.eq('Yes'), 'smoking_location'].eq('Inside').all()

## Specify the manuscript category order

Keep categories and their order stable, including Not recorded. Legal tenure and respondent tenure status remain separate rows, and absent categories are represented by zero rather than dropped.

In [ ]:
# Characteristics shown in Table 1 and the fixed order of their categories.
# Absent categories are reported as zero rather than dropped.
cats = {
    'Respondent tenure status': q.respondent_status,
    'Legal tenure as recorded': q.tenure,
    'Cooking fuel': q.cooking,
    'Shower type': q.shower_type,
    'Thermostat location': loc,
    'Household pets': q.pets,
    'Smoking or vaping': smoke,
    'Summer overheating reported': q.overheating,
    'Air conditioning installed': q.air_conditioning,
}
order = {
    'Respondent tenure status': ['Tenant', 'Homeowner', 'Not recorded'],
    'Legal tenure as recorded': ['Freehold', 'Not recorded'],
    'Cooking fuel': ['Gas', 'Electric', 'Not recorded'],
    'Shower type': ['Electric', 'Mixer', 'Mixer and electric', 'Not recorded'],
    'Thermostat location': ['Hallway or hall', 'Living or dining room', 'Not recorded'],
    'Household pets': ['Yes', 'No', 'Not recorded'],
    'Smoking or vaping': ['No', 'Yes, indoors', 'Not recorded'],
    'Summer overheating reported': ['No', 'Yes', 'Not recorded'],
    'Air conditioning installed': ['No', 'Not recorded'],
}

## Calculate available-case counts and numeric summaries

Categorical counts include the explicit missing category. Numeric medians, quartiles, and ranges use available responses and report the valid n for each characteristic. No answer is inferred from another household.

In [ ]:
# Categorical counts (Table 1), including the explicit "Not recorded" category.
rows = []
for name, values in cats.items():
    values = values.fillna('Not recorded').replace({'Mixer and Electric': 'Mixer and electric'})
    counts = values.value_counts()
    assert set(counts.index) <= set(order[name]), (name, counts.to_dict())
    for category in order[name]:
        rows.append({'Characteristic': name, 'Category': category, 'Homes (n)': int(counts.get(category, 0))})

# Continuous characteristics: available-case n, median, quartiles and range.
numeric = []
for label, c in [('Household size', 'occupants'), ('Thermostat setpoint (°C)', 'thermostat'),
                 ('Showering frequency (times/week)', 'showers'), ('Bathing frequency (times/week)', 'baths')]:
    s = q[c].dropna()
    numeric.append(dict(Characteristic=label, n=len(s), median=s.median(), q25=s.quantile(.25),
                        q75=s.quantile(.75), min=s.min(), max=s.max()))

## Save Tables 1 and S4 with linkage evidence

Expect 23 retained homes, 22 matched assessment records, and 11 matched project-survey records. Save the source worksheet row numbers so a reviewer can trace each displayed value. Detailed linkage data remain local to the project.

In [ ]:
pd.DataFrame(rows).to_csv(out / 'tables/Table_1_categorical.csv', index=False)
pd.DataFrame(numeric).to_csv(out / 'tables/Table_1_numeric.csv', index=False)
pd.DataFrame(housing).to_csv(out / 'tables/Table_S4.csv', index=False)
q.to_csv(out / 'data/questionnaire_by_home.csv', index=False)
pd.DataFrame(linkage).to_csv(out / 'tables/questionnaire_linkage.csv', index=False)

# 22 homes have an assessment record and 11 a project-survey record.
assert q.assessment_row.notna().sum() == 22 and sum(r['project_survey_row'] is not None for r in linkage) == 11
dump_json(out / 'logs/questionnaire_checks.json', dict(
    homes=23, assessment_matches=22, project_matches=11, missing_data_imputed=False,
    tenure_conflicts=[r['home'] for r in linkage if r['tenure_conflict']]))
print('Tables 1 and S4 rebuilt from source questionnaires: 22 assessment and 11 project-survey matches.')

## Interpret the outputs

Review the saved CSV files and JSON checks in the selected output directory. Keep the settings in `analysis_settings.json`
fixed when reproducing this manuscript. If a source-data assertion fails, inspect the named records before changing
any thresholds. `README.md` documents the order of all manuscript stages.